# Prompt 24: Performance Tuning Techniques
## Databricks Certified Associate Developer for Apache Spark
### Topic 4 — Troubleshooting and Tuning Spark Applications (10%)

---

## Spark UI Navigation

The Spark Web UI is available at `http://<driver-host>:4040` while a Spark application is running (or via the Databricks cluster UI). Each tab exposes a different layer of the execution hierarchy.

| Tab | What it shows | Key use |
|-----|---------------|---------|
| **Jobs** | All jobs triggered by actions; status (running/succeeded/failed); duration | Identify which job is slow or failed |
| **Stages** | Each job's stages; shuffle read/write bytes; skipped stages (cache hit) | Identify expensive shuffle; compare stage times |
| **Tasks** | Per-task metrics within a stage: duration, GC time, shuffle read/write, input size | **Detect data skew** — compare median vs max task duration |
| **SQL** | DAG visualization of Spark SQL / DataFrame query plans | Match plan nodes to code; see if BroadcastHashJoin fired |
| **Storage** | Cached RDDs/DataFrames: storage level, memory used, disk used, fraction cached | Verify cache hit; find uncached DataFrames taking memory |
| **Environment** | All Spark config properties in effect; Java/Scala versions; classpath | Verify your `spark.conf.set()` changes took effect |

### Navigating a Failure
```
Jobs tab → click failed job
  → Stages tab shows which stage failed (red)
    → click failed stage
      → Tasks tab shows failed task (red row)
        → expand task row → "Full Exception" link → stack trace
```

---

## Memory Model

```
JVM heap per executor
├── Spark Memory (spark.memory.fraction = 0.6 of heap)    ← tunable
│   ├── Storage Memory  (cache, broadcast)               ← dynamic split
│   └── Execution Memory (shuffle, sort, aggregation)    ← dynamic split
└── User Memory (0.4 of heap) — user data structures, UDFs

Off-heap
└── Memory Overhead (spark.executor.memoryOverhead)       ← JVM overhead, native libs
```

| Config | Default | Description |
|--------|---------|-------------|
| `spark.driver.memory` | 1g | Driver JVM heap size |
| `spark.executor.memory` | 1g | Executor JVM heap size |
| `spark.executor.memoryOverhead` | max(384MB, 10% of executor memory) | Off-heap overhead (native libs, JVM overhead) |
| `spark.memory.fraction` | 0.6 | Fraction of heap used by Spark (storage + execution) |
| `spark.memory.storageFraction` | 0.5 | Fraction of Spark memory reserved for storage (rest for execution) |

**Storage vs Execution memory** share the Spark memory pool dynamically — execution can borrow from storage (evicting cached data) and vice versa.

---

## Key Configuration Properties

| Config | Default | When to change |
|--------|---------|----------------|
| `spark.executor.cores` | depends on cluster manager | Number of cores per executor; more cores = more parallelism per executor |
| `spark.default.parallelism` | depends on cluster | Default RDD partition count for `sc.parallelize`, joins, groupBy without shuffle partition setting |
| `spark.sql.shuffle.partitions` | **200** | Number of partitions created after a shuffle (sort/groupBy/join); reduce for small datasets (e.g., 4–20) to avoid 200 tiny tasks |
| `spark.sql.autoBroadcastJoinThreshold` | **10MB** | Max size of a relation that Spark will auto-broadcast in a join; set to `-1` to disable; increase to allow larger broadcast |
| `spark.sql.adaptive.enabled` | true (Spark 3.2+) | Enable Adaptive Query Execution (AQE) — runtime optimisation |

**Exam tip:** The default of `spark.sql.shuffle.partitions = 200` is frequently tested. On a small dataset it creates 200 tiny partitions, wasting overhead. Always tune this to roughly `2–4 × number of cores`.

---

## Caching and Persistence

```python
from pyspark import StorageLevel

df.cache()                              # shortcut for MEMORY_AND_DISK
df.persist()                            # same as cache()
df.persist(StorageLevel.MEMORY_ONLY)    # fastest reads; evicts under pressure → recompute
df.persist(StorageLevel.MEMORY_AND_DISK)# spills to disk if no room in memory
df.persist(StorageLevel.MEMORY_ONLY_SER)# Java-serialised (smaller footprint, slower reads)
df.persist(StorageLevel.DISK_ONLY)      # always on disk; slowest

df.unpersist()                          # free cached data — call when done
```

| StorageLevel | Memory used | CPU cost | Notes |
|---|---|---|---|
| `MEMORY_ONLY` | Deserialized objects | Low | Fastest; risks OOM |
| `MEMORY_AND_DISK` | Memory then disk | Low | Default for `cache()` |
| `MEMORY_ONLY_SER` | Serialized in memory | Medium | Smaller footprint |
| `DISK_ONLY` | Disk | High (I/O) | Slowest; no memory usage |

**When to cache:**
- A DataFrame is used in multiple actions (avoids recomputation)
- Iterative algorithms (ML training loops)
- After an expensive transformation

**When NOT to cache:**
- DataFrame is used only once
- Dataset is too large for available memory (causes eviction thrash)
- Data is trivially recomputed

---

## Shuffle Reduction

Shuffles are the most expensive Spark operations — they require moving data across network to different executors. Strategies to minimise:

1. **Broadcast small DataFrames** — eliminates the shuffle entirely for joins:
   ```python
   from pyspark.sql.functions import broadcast
   large_df.join(broadcast(small_df), on='key')  # no shuffle needed
   ```

2. **Repartition before joins on join key** — if two large DataFrames will be joined frequently on the same key, pre-partition both by that key to co-locate matching records:
   ```python
   df_orders   = df_orders.repartition(200, 'customer_id')
   df_customers = df_customers.repartition(200, 'customer_id')
   df_orders.join(df_customers, on='customer_id')  # shuffle already done
   ```

3. **Reduce `spark.sql.shuffle.partitions`** for small datasets — fewer partitions = less task scheduling overhead.

4. **Avoid unnecessary wide transformations** — `groupBy`, `join`, `distinct`, `orderBy` all trigger shuffles. Filter and project early to reduce data volume before the shuffle.

---

## Adaptive Query Execution (AQE)

AQE re-optimises the query plan at **runtime** using actual statistics collected during execution.

```python
# Enable AQE (default True in Spark 3.2+)
spark.conf.set('spark.sql.adaptive.enabled', 'true')
```

| Feature | What it does |
|---------|-------------|
| **Coalescing shuffle partitions** | After a shuffle, merges tiny partitions into fewer, larger partitions — avoids 200 tiny tasks on small datasets |
| **Optimising skew joins** | Detects skewed partitions in a sort-merge join and splits them to run in parallel |
| **Switching join strategies** | Can switch from SortMergeJoin to BroadcastHashJoin at runtime if one side turns out to be small |

**Related configs:**
```python
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')  # default true
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')            # default true
spark.conf.set('spark.sql.adaptive.advisoryPartitionSizeInBytes', '64MB') # target coalesced size
```

---

## Data Skew: Detection and Salting

**Detection:** In Spark UI → Stages → Tasks tab, compare **median task duration** vs **max task duration**. If max >> median (e.g., max = 10 min, median = 30 sec), data is skewed.

**Root cause:** A small number of partition keys appear very frequently (e.g., `country = 'US'` dominates).

**Fix 1: AQE skew join optimisation** (automatic if AQE is enabled — Spark 3.0+)

**Fix 2: Manual salting** for joins:
```python
import pyspark.sql.functions as F
import random

SALT_FACTOR = 10

# Salt the large skewed DataFrame
df_large_salted = df_large.withColumn(
    'salt_key',
    F.concat(col('join_key'), F.lit('_'), (F.rand() * SALT_FACTOR).cast('int'))
)

# Replicate the small DataFrame for each salt value
salt_range = spark.range(SALT_FACTOR).withColumnRenamed('id', 'salt')
df_small_replicated = df_small.crossJoin(salt_range).withColumn(
    'salt_key',
    F.concat(col('join_key'), F.lit('_'), col('salt').cast('string'))
)

# Join on the salted key
result = df_large_salted.join(df_small_replicated, on='salt_key', how='inner') \
    .drop('salt_key', 'salt')
```

In [ ]:
# Cell 1: SparkSession with tuning configs set at startup
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, rand, concat, lit, when, count, sum as _sum, avg
from pyspark.sql.types import IntegerType, StringType, DoubleType, StructType, StructField
from pyspark import StorageLevel
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName('PerformanceTuning') \
    .master('local[4]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.autoBroadcastJoinThreshold', '10MB') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Verify configs took effect
print('=== Effective configuration ===')
for key in [
    'spark.sql.shuffle.partitions',
    'spark.sql.adaptive.enabled',
    'spark.sql.autoBroadcastJoinThreshold',
    'spark.driver.memory',
]:
    print(f'  {key} = {spark.conf.get(key)}')

# ---------------------------------------------------------------
# Changing config at runtime (overrides builder config)
# ---------------------------------------------------------------
spark.conf.set('spark.sql.shuffle.partitions', '4')
print('\nAfter runtime override:')
print(f'  spark.sql.shuffle.partitions = {spark.conf.get("spark.sql.shuffle.partitions")}')

In [ ]:
# Cell 2: Caching and persistence — effect on query plans
import time

# Create a DataFrame that represents an expensive computation
large_df = spark.range(0, 1_000_000) \
    .withColumn('value', (col('id') % 100).cast(DoubleType())) \
    .withColumn('category', when(col('id') % 3 == 0, 'A')
                             .when(col('id') % 3 == 1, 'B')
                             .otherwise('C'))

# Without cache: computes the transformation twice
t0 = time.time()
c1 = large_df.count()
c2 = large_df.groupBy('category').agg(_sum('value')).collect()
t_no_cache = time.time() - t0
print(f'Without cache: {t_no_cache:.2f}s (transformation computed twice)')

# With cache: computes transformation once, reuses in memory
large_df.cache()
_ = large_df.count()   # materialize the cache
t0 = time.time()
c3 = large_df.count()
c4 = large_df.groupBy('category').agg(_sum('value')).collect()
t_with_cache = time.time() - t0
print(f'With cache:    {t_with_cache:.2f}s (transformation read from memory)')

# Check the Storage tab (programmatically)
print(f'\nDataFrame cached: {large_df.is_cached}')

# Always unpersist when done
large_df.unpersist()
print(f'After unpersist — is_cached: {large_df.is_cached}')

# ---------------------------------------------------------------
# Persist with explicit StorageLevel
# ---------------------------------------------------------------
print('\n=== StorageLevel options ===')
for level_name, level in [
    ('MEMORY_ONLY',     StorageLevel.MEMORY_ONLY),
    ('MEMORY_AND_DISK', StorageLevel.MEMORY_AND_DISK),
    ('DISK_ONLY',       StorageLevel.DISK_ONLY),
]:
    df_temp = spark.range(1000).persist(level)
    df_temp.count()  # materialise
    print(f'  {level_name}: is_cached={df_temp.is_cached}, storageLevel={df_temp.storageLevel}')
    df_temp.unpersist()

In [ ]:
# Cell 3: Shuffle partitions and join strategy

# Sample data — orders (large) and customers (small reference table)
orders = spark.range(0, 50_000) \
    .withColumn('customer_id', (col('id') % 100).cast(IntegerType())) \
    .withColumn('amount', (rand() * 500).cast(DoubleType()))

customers = spark.createDataFrame(
    [(i, f'Customer_{i}', 'US' if i % 2 == 0 else 'CA') for i in range(100)],
    ['customer_id', 'name', 'country']
)

# ---------------------------------------------------------------
# Default join — may or may not broadcast depending on threshold
# ---------------------------------------------------------------
print('=== Join plan without explicit broadcast hint ===')
df_joined = orders.join(customers, on='customer_id', how='inner')
df_joined.explain()   # look for BroadcastHashJoin vs SortMergeJoin

# ---------------------------------------------------------------
# Explicit broadcast hint — forces BroadcastHashJoin
# ---------------------------------------------------------------
print('=== Join plan WITH broadcast hint ===')
df_broadcast_joined = orders.join(broadcast(customers), on='customer_id', how='inner')
df_broadcast_joined.explain()   # must show BroadcastHashJoin

# Show result
df_broadcast_joined.groupBy('country').agg(
    count('id').alias('order_count'),
    _sum('amount').alias('total_revenue')
).show()

# ---------------------------------------------------------------
# Effect of shuffle partitions on groupBy output
# ---------------------------------------------------------------
spark.conf.set('spark.sql.shuffle.partitions', '200')  # default
result_200 = orders.groupBy('customer_id').agg(_sum('amount'))
print(f'With 200 shuffle partitions: {result_200.rdd.getNumPartitions()} result partitions')

spark.conf.set('spark.sql.shuffle.partitions', '8')    # tuned for small dataset
result_8 = orders.groupBy('customer_id').agg(_sum('amount'))
print(f'With 8 shuffle partitions:   {result_8.rdd.getNumPartitions()} result partitions')

In [ ]:
# Cell 4: Detecting and fixing data skew via salting

# Create a skewed DataFrame: 80% of rows have customer_id = 1
from pyspark.sql.functions import floor

skewed_data = (
    [(1, float(i)) for i in range(8000)] +   # skewed key
    [(j, float(j)) for j in range(2, 102) for _ in range(20)]  # normal keys
)
df_skewed = spark.createDataFrame(skewed_data, ['customer_id', 'amount'])
print('=== Skewed data distribution ===')
df_skewed.groupBy('customer_id').count().orderBy(col('count').desc()).show(5)

# ---------------------------------------------------------------
# Skewed join simulation
# ---------------------------------------------------------------
df_ref = spark.createDataFrame(
    [(j, f'Customer_{j}') for j in range(1, 102)],
    ['customer_id', 'name']
)

# Without AQE skew handling — customer_id=1 goes to one task
print('=== Join physical plan (skewed, no salting) ===')
df_skewed.join(df_ref, on='customer_id', how='inner').explain()

# ---------------------------------------------------------------
# Salting technique to spread the skewed key
# ---------------------------------------------------------------
SALT_FACTOR = 10

# Add random salt to the large skewed DataFrame
df_large_salted = df_skewed.withColumn(
    'salt_key',
    F.concat(
        col('customer_id').cast(StringType()),
        F.lit('_'),
        (F.rand() * SALT_FACTOR).cast(IntegerType()).cast(StringType())
    )
)

# Replicate the small reference table for each salt value
salt_df = spark.range(SALT_FACTOR).withColumnRenamed('id', 'salt')
df_ref_replicated = df_ref.crossJoin(salt_df).withColumn(
    'salt_key',
    F.concat(
        col('customer_id').cast(StringType()),
        F.lit('_'),
        col('salt').cast(StringType())
    )
)

# Join on salt_key — skewed key is now spread across 10 tasks
df_salted_result = df_large_salted.join(
    df_ref_replicated, on='salt_key', how='inner'
).drop('salt_key', 'salt').drop(df_ref_replicated.customer_id)

print('=== Salted join result (first 5 rows) ===')
df_salted_result.show(5)

# ---------------------------------------------------------------
# AQE as automatic alternative
# ---------------------------------------------------------------
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
print('AQE skew join enabled — Spark will automatically split skewed partitions at runtime.')
print('AQE config:', spark.conf.get('spark.sql.adaptive.enabled'))

In [ ]:
# Cell 5: Reading the explain() plan for tuning insights

df_orders = spark.range(0, 10_000) \
    .withColumn('customer_id', (col('id') % 200).cast(IntegerType())) \
    .withColumn('amount', (rand() * 500).cast(DoubleType())) \
    .withColumn('region', when(col('id') % 2 == 0, 'West').otherwise('East'))

# ---------------------------------------------------------------
# Plan analysis: filter pushed down vs not
# ---------------------------------------------------------------
print('=== Formatted explain — filter before groupBy (good: filter pushdown) ===')
df_orders.filter(col('region') == 'West') \
         .groupBy('customer_id').agg(_sum('amount').alias('total')) \
         .explain('formatted')

# ---------------------------------------------------------------
# Different explain modes
# ---------------------------------------------------------------
simple_query = df_orders.groupBy('region').agg(count('id').alias('cnt'))

print('=== explain() — simple (physical plan only) ===')
simple_query.explain()

print('\n=== explain("extended") — logical + physical ===')
simple_query.explain('extended')

print('\n=== explain("cost") — includes row count estimates ===')
simple_query.explain('cost')

# ---------------------------------------------------------------
# What to look for in physical plan for tuning
# ---------------------------------------------------------------
print('\n=== PLAN READING GUIDE ===')
print('BroadcastHashJoin   → small table was broadcast; efficient')
print('SortMergeJoin       → both sides shuffled; expensive for large tables')
print('BroadcastExchange   → a broadcast is happening')
print('Exchange hashpartitioning → shuffle is happening (check shuffle bytes in Stages tab)')
print('Filter              → if above Scan: good (pushed down to data source)')
print('Filter              → if below HashAggregate: not pushed down; data was read first')
print('ColumnarToRow       → reading Parquet columnar data and converting to rows')
print('AdaptiveSparkPlan   → AQE is wrapping this plan (re-optimised at runtime)')

spark.stop()
print('\nDone.')

## Real-World Scenarios

<details>
<summary>Scenario 1: Reducing 200 empty shuffle partitions on a small dataset</summary>

**Situation:**
A team processes a small daily delta file (~5,000 rows). A `groupBy` aggregation takes 45 seconds due to the default `spark.sql.shuffle.partitions = 200`. The Stages tab shows 195 tasks completing in under 1ms (empty partitions) and 5 tasks doing real work.

**Fix:**
```python
# Before any groupBy/join in the session:
spark.conf.set('spark.sql.shuffle.partitions', '8')

# Or set at session creation:
spark = SparkSession.builder \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()

# Or enable AQE to handle it automatically:
spark.conf.set('spark.sql.adaptive.enabled', 'true')
# AQE will coalesce the 200 post-shuffle partitions into fewer, larger ones
```

**Outcome:** Stage duration drops from 45s to ~3s. The Stages tab now shows 8 tasks with balanced durations.

**Exam Sub-topic:** `spark.sql.shuffle.partitions`; AQE coalescing; Spark UI Stages tab
</details>

<details>
<summary>Scenario 2: Forcing a broadcast join when autoBroadcastJoinThreshold is not triggered</summary>

**Situation:**
A join between a 50GB orders table and a 15MB currency reference table is using SortMergeJoin (visible in Spark UI SQL tab). The 15MB table exceeds the default `autoBroadcastJoinThreshold` of 10MB so Spark didn't auto-broadcast it. Each job stage writes hundreds of MBs of shuffle data.

**Two solutions:**
```python
# Option A: Raise the threshold
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '20MB')

# Option B: Use explicit broadcast hint (doesn't depend on size threshold)
from pyspark.sql.functions import broadcast
result = df_orders.join(broadcast(df_currency), on='currency_code', how='left')

# Verify: explain() must show BroadcastHashJoin
result.explain()
# Look for: BroadcastHashJoin [...], BuildRight (the broadcast side)
```

**Outcome:** Shuffle data drops to near zero. Join stage runtime reduces by ~80%.

**Exam Sub-topic:** `spark.sql.autoBroadcastJoinThreshold`; broadcast hint; reading `explain()` for join type
</details>

<details>
<summary>Scenario 3: Caching a frequently reused DataFrame in an iterative pipeline</summary>

**Situation:**
An ETL pipeline reads a large Parquet file, applies 10 filter + join transformations, then writes 5 different output files from the same intermediate DataFrame. Without caching, the Parquet file is read and the transformations are re-executed 5 times (once per write action).

**Fix:**
```python
# Read + transform once
df_base = spark.read.parquet('/data/events') \
    .filter(col('event_type').isin(['click', 'view', 'purchase'])) \
    .withColumn('event_date', to_date(col('event_ts')))

# Cache before the 5 writes — transformation computed once, stored in memory
df_base.cache()
df_base.count()  # trigger materialisation (optional but makes first write faster)

# All 5 writes read from cache, not re-executing the Parquet read + filter + join
df_base.filter(col('event_type') == 'click').write.parquet('/output/clicks')
df_base.filter(col('event_type') == 'view').write.parquet('/output/views')
df_base.filter(col('event_type') == 'purchase').write.parquet('/output/purchases')
df_base.groupBy('event_date').count().write.parquet('/output/daily_counts')
df_base.agg(count('event_id'), avg('session_duration')).write.parquet('/output/summary')

# Free memory when done
df_base.unpersist()
```

**Outcome:** Parquet read executed once instead of 5 times. Verify in Spark UI Storage tab — cached DataFrame shows memory footprint.

**Exam Sub-topic:** `cache()` vs `persist()`; `unpersist()`; Spark UI Storage tab
</details>

<details>
<summary>Scenario 4: Identifying data skew in the Spark UI and applying salting</summary>

**Situation:**
A join between a 100GB transaction table and a 1GB merchant table is slow. The Spark UI Tasks tab shows: median task duration = 15s, max task duration = 47 minutes. The slow task corresponds to merchant_id = 'AMAZON' which accounts for 60% of all transactions.

**Diagnosis:**
- Tasks tab: sort by Duration descending — the outlier task has much higher shuffle read bytes
- A single partition holds all AMAZON rows → processed by one executor core

**Fix — Manual salting:**
```python
SALT = 20  # spread AMAZON across 20 tasks

# Salt the large transactions table
df_tx_salted = df_transactions.withColumn(
    'salted_merchant',
    F.concat(col('merchant_id'), F.lit('_'), (F.rand() * SALT).cast('int').cast('string'))
)

# Replicate the merchant reference table
df_merchant_replicated = df_merchants \
    .crossJoin(spark.range(SALT).withColumnRenamed('id', 's')) \
    .withColumn(
        'salted_merchant',
        F.concat(col('merchant_id'), F.lit('_'), col('s').cast('string'))
    )

result = df_tx_salted.join(df_merchant_replicated, on='salted_merchant') \
    .drop('salted_merchant', 's')

# Alternative: enable AQE skew join (automatic, no code change)
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
```

**Outcome:** AMAZON rows spread across 20 tasks. Max task duration drops to ~2.5 minutes. Overall stage runtime reduces by ~95%.

**Exam Sub-topic:** Data skew detection in Tasks tab; salting; AQE skew join
</details>

<details>
<summary>Scenario 5: Navigating Spark UI from a failed job to the root cause error message</summary>

**Situation:**
A production Spark job fails with a generic `SparkException`. The developer needs to find the actual error message (e.g., a Python UDF `AttributeError`) buried in the distributed log output.

**Navigation path:**
```
1. Spark UI → Jobs tab
   → Find the FAILED job (red status)
   → Click the job link

2. → Stages tab (now filtered to this job)
   → Find the FAILED stage (red)
   → Click the stage link

3. → Tasks tab within the failed stage
   → Sort by Status = FAILED
   → Click the failed task row
   → Expand: "Full Exception" or "Errors" column
   → Read the Python traceback

4. Alternatively: stderr logs of the executor
   → In the Tasks tab, click the executor's "Logs" link
   → Search for ERROR or the exception class name
```

**Code fix example (once error found — UDF AttributeError on None):**
```python
# Error: AttributeError: 'NoneType' object has no attribute 'upper'
# Fix: add None guard
@udf(StringType())
def safe_upper(s):
    return s.upper() if s is not None else None
```

**Exam Sub-topic:** Spark UI Jobs/Stages/Tasks navigation; reading executor logs; UDF null safety
</details>

## Exam Cheat Sheet

| Question | Answer |
|----------|--------|
| Default `spark.sql.shuffle.partitions` | **200** |
| When to reduce shuffle partitions | Small datasets — reduces empty task overhead |
| Default `autoBroadcastJoinThreshold` | **10MB** |
| Set at build time vs runtime | `.config()` at build, `spark.conf.set()` at runtime |
| Disable broadcast altogether | `spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')` |
| `cache()` is shortcut for | `persist(StorageLevel.MEMORY_AND_DISK)` |
| Free cached data | `df.unpersist()` |
| Verify cache hit | Spark UI Storage tab; `df.is_cached` |
| Enable AQE | `spark.conf.set('spark.sql.adaptive.enabled', 'true')` |
| AQE default in Spark 3.2+ | **True** |
| AQE coalesces | Small shuffle partitions after a shuffle |
| AQE skew join | Splits skewed sort-merge join partitions at runtime |
| Detect skew | Spark UI Tasks tab — compare median vs max task duration |
| Fix skew manually | Salting — add random suffix to join key on both sides |
| `explain()` shows BroadcastHashJoin | Broadcast join was used |
| `explain()` shows SortMergeJoin | Both sides were shuffled |
| `explain('formatted')` | Readable indented physical plan |
| `explain('extended')` | Logical + physical plan |
| `explain('cost')` | Includes row count and size estimates |
| UI tab for config values | **Environment** tab |
| UI tab for cache status | **Storage** tab |
| UI tab for per-task skew | **Tasks** (within a Stage) |

---

## Practice Q&A

<details>
<summary>Q1: A groupBy on a 500-row DataFrame takes 30 seconds. What is the most likely cause and fix?</summary>

**A:** The most likely cause is the default `spark.sql.shuffle.partitions = 200`. Spark creates 200 post-shuffle partitions for a tiny dataset, spawning 200 tasks — most of which do zero work but still incur scheduling overhead.

**Fix:**
```python
spark.conf.set('spark.sql.shuffle.partitions', '4')   # match to number of cores
# or enable AQE — it will auto-coalesce to a few partitions:
spark.conf.set('spark.sql.adaptive.enabled', 'true')
```
</details>

<details>
<summary>Q2: What is the difference between cache() and persist(StorageLevel.MEMORY_ONLY)?</summary>

**A:** `cache()` is a shortcut for `persist(StorageLevel.MEMORY_AND_DISK)`. It first stores data in JVM heap memory; if memory is insufficient, it **spills to disk** rather than recomputing.

`persist(StorageLevel.MEMORY_ONLY)` stores only in memory. If there isn't enough space, Spark **drops partitions and recomputes** them on demand — no disk fallback.

Use `MEMORY_AND_DISK` (i.e., `cache()`) when the dataset might not fit in memory. Use `MEMORY_ONLY` when the dataset definitely fits and you want the fastest possible reads.
</details>

<details>
<summary>Q3: In the Spark UI, how do you determine whether a query used a broadcast join or a sort-merge join?</summary>

**A:** Navigate to the **SQL tab** in the Spark UI and click on the query. The DAG visualization shows the physical plan as a graphical tree. Look for node labels:
- **`BroadcastHashJoin`** — one side was broadcast; no shuffle for the join itself
- **`SortMergeJoin`** — both sides were shuffled and sorted before joining

Alternatively, run `df.explain()` in code. The physical plan will contain `BroadcastHashJoin` or `SortMergeJoin` as plan node names.
</details>

<details>
<summary>Q4: What three things does AQE automatically optimise at runtime?</summary>

**A:**
1. **Coalescing small shuffle partitions** — After a shuffle, AQE merges tiny partitions into fewer, right-sized partitions. Eliminates the 200-empty-partitions problem without manually setting `spark.sql.shuffle.partitions`.
2. **Skew join optimisation** — Detects partitions in a sort-merge join that are much larger than others and automatically splits them so they can be processed in parallel.
3. **Join strategy switching** — Can change a planned SortMergeJoin to a BroadcastHashJoin at runtime if the actual data size (after filtering) turns out to be small enough to broadcast.

All three are enabled by default when `spark.sql.adaptive.enabled = true` (Spark 3.2+).
</details>

<details>
<summary>Q5: How do you detect data skew in the Spark UI and what metric is the key indicator?</summary>

**A:** Navigate to: Spark UI → **Jobs** → select the slow job → **Stages** → select the slow stage → **Tasks** tab.

Sort tasks by **Duration** descending. The key indicator is the gap between **median task duration** and **max task duration**:
- If median = 5s and max = 5min → severe skew (one partition has most of the data)
- Also check the **Shuffle Read Size** column — the skewed task will have disproportionately large shuffle reads

The summary row at the top of the Tasks table shows min/median/max/75th percentile — compare these. A large ratio (max >> 75th percentile) confirms skew.
</details>